# Ingesta y Transformación de Datos IoT en Microsoft Fabric

## Objetivo
Cargar el fichero **`sensores_x.csv`** desde OneLake, aplicar transformaciones de limpieza y guardar el resultado en una tabla Delta del Lakehouse.

El ejercicio se realiza de **dos formas**:
1. **Notebook (PySpark)** — código explícito, controlado y reproducible
2. **Dataflow Gen2 (Power Query)** — interfaz gráfica, sin código

---
## Parte 1 — Notebook PySpark

In [ ]:
from pyspark.sql.functions import col, regexp_replace, to_timestamp

# 1. Lectura
df = spark.read.format("csv").option("header", "true").load("Files/bronze/sensores_2026.csv")

# 2. Transformaciones (Solo quitamos el % y convertimos a entero/double)
df_clean = df.filter(col("ID_Sensor").isNotNull()) \
    .withColumn("Timestamp", to_timestamp(col("Timestamp"))) \
    .withColumn("Valor_PPM", col("Valor_PPM").cast("double")).fillna(0, subset=["Valor_PPM"]) \
    .withColumn("Latitud", col("Latitud").cast("double")) \
    .withColumn("Longitud", col("Longitud").cast("double")) \
    .withColumn("Bateria", 
        regexp_replace(col("Estado_Bateria"), "%", "").cast("bigint")
    ) \
    .select("ID_Sensor", "Timestamp", "Tipo_Contaminante", "Valor_PPM", "Latitud", "Longitud", "Bateria")

# 3. Guardar — overwrite mode + overwriteSchema para permitir cambios en el esquema
df_clean.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .save("Tables/silver/pyspark_sensor_data")

---
## Parte 2 — Dataflow Gen2 (Power Query en Fabric)

Replica la misma lógica de transformación usando la interfaz visual de **Dataflow Gen2** en Microsoft Fabric.  
No es necesario escribir código: cada paso se configura mediante menús y asistentes.

---

### Paso 1 — Crear el Dataflow Gen2

1. Abre tu **Workspace** en Microsoft Fabric.
2. Haz clic en **+ Nuevo** → **Dataflow Gen2**.
3. Se abrirá el editor de Power Query.

---

### Paso 2 — Obtener el fichero CSV

1. En el editor de Power Query, haz clic en **Obtener datos**.
2. Selecciona **OneLake** (o **Azure Data Lake Storage Gen2** si accedes directamente).
3. Navega hasta tu Lakehouse → carpeta **Files** → selecciona **`sensores_madrid_2026.csv`**.
4. En la vista previa, marca la opción **La primera fila como encabezado** y confirma.

---

### Paso 3 — Filtrar filas nulas en `ID_Sensor`

1. Haz clic en el icono de filtro de la columna **`ID_Sensor`**.
2. Desmarca la opción **(null)** y **vacío** (si aparece).
3. Haz clic en **Aceptar**.

> Equivale a `.filter(col("ID_Sensor").isNotNull())` en PySpark.

---

### Paso 4 — Cambiar tipo de `Timestamp`

1. Selecciona la columna **`Timestamp`**.
2. En la pestaña **Transformar**, haz clic en **Tipo de datos** → **Fecha/hora**.
3. Confirma el cambio si se solicita.

> Equivale a `to_timestamp(col("Timestamp"))` en PySpark.

---

### Paso 5 — Limpiar y castear `Valor_PPM`

1. Selecciona la columna **`Valor_PPM`**.
2. En la pestaña **Transformar**, haz clic en **Reemplazar errores** → introduce `0`.
3. A continuación, ve a **Transformar** → **Tipo de datos** → **Número decimal**.
4. Vuelve a aplicar **Reemplazar valores**: Valor a buscar = `null`, Reemplazar por = `0`.

> Equivale a `.cast("double").fillna(0, subset=["Valor_PPM"])` en PySpark.

---

### Paso 6 — Castear `Latitud` y `Longitud`

1. Selecciona la columna **`Latitud`**.
2. Ve a **Transformar** → **Tipo de datos** → **Número decimal**.
3. Repite los mismos pasos con la columna **`Longitud`**.

> Equivale a `.withColumn("Latitud", col("Latitud").cast("double"))` y lo mismo para `Longitud` en PySpark.

---

### Paso 7 — Transformar `Estado_Bateria` → columna `Bateria`

El campo contiene valores con formato `"XX%"`. Solo hay que eliminar el símbolo `%` y convertir a número entero largo.

**7a. Extraer el número eliminando el símbolo `%`:**
1. Selecciona la columna **`Estado_Bateria`**.
2. Ve a **Transformar → Extraer → Texto antes del delimitador** → introduce `%` → Aceptar.

**7b. Convertir a número entero:**
1. Con la columna seleccionada, ve a **Transformar** → **Tipo de datos** → **Número entero**.

**7c. Renombrar la columna:**
1. Doble clic sobre el encabezado → escribe **`Bateria`** → Enter.

> Equivale a `regexp_replace(col("Estado_Bateria"), "%", "").cast("bigint")` en PySpark.

---

### Paso 8 — Eliminar columnas innecesarias

1. Selecciona todas las columnas que **no** formen parte del resultado final.  
   Mantén únicamente: `ID_Sensor`, `Timestamp`, `Tipo_Contaminante`, `Valor_PPM`, `Latitud`, `Longitud`, `Bateria`.
2. Haz clic derecho sobre las columnas a eliminar → **Quitar columnas**.

---

### Paso 9 — Configurar el destino de datos

1. En la parte inferior del editor, busca el panel **Destino de datos** (o ve a **Inicio → Agregar destino de datos**).
2. Selecciona **Lakehouse**.
3. Elige tu Lakehouse y en el campo de tabla escribe: **`dataflow_sensors_data`**.
4. Configura el método de actualización: **Reemplazar** — esto sobrescribe los datos **y** el esquema de la tabla si ha cambiado.
5. Haz clic en **Siguiente** y luego en **Guardar configuración**.

> En el notebook PySpark esto equivale a `mode("overwrite")` + `.option("overwriteSchema", "true")`.

---

### Paso 10 — Publicar y ejecutar

1. Haz clic en **Publicar** (esquina superior derecha).
2. El Dataflow se guardará en el Workspace.
3. Para ejecutarlo manualmente: en el Workspace, busca el Dataflow → **··· (más opciones)** → **Actualizar ahora**.
4. Verifica el resultado abriendo el Lakehouse: debería aparecer la tabla **`dataflow_sensor_data`** con las 7 columnas.

---
## Resumen comparativo

| Característica | Notebook (PySpark) | Dataflow Gen2 (Power Query) |
|---|---|---|
| **Interfaz** | Código Python | Visual / sin código |
| **Control** | Total | Configuración guiada |
| **Reproducibilidad** | Alta (código versionable) | Media (exportable a JSON) |
| **Escalabilidad** | Alta (Spark distribuido) | Moderada |
| **Tabla destino** | `pyspark_sensor_data` | `dataflow_sensor_data` |
| **Modo escritura** | `overwrite` + `overwriteSchema=true` | Reemplazar (datos y esquema) |
| **Tipo `Bateria`** | `bigint` | Número entero |
| **Ideal para** | Ingenieros de datos | Analistas / usuarios de negocio |

Ambos enfoques producen el mismo resultado final en el Lakehouse. Elige el que mejor se adapte al perfil del equipo y a los requisitos del proyecto.